In [1]:
print('hi')

hi


# LLM 

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
groq_api_key = os.getenv('groq_api_key')
cohere_api_key = os.getenv('cohere_api_key')
groq_api_key[:2], cohere_api_key[:2]

('gs', 'ym')

In [3]:
os.environ['groq_api_key'] = groq_api_key

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model = 'llama-3.1-8b-instant'
)

In [5]:
llm.invoke('what is gen ai')

AIMessage(content='General AI (Gen AI) refers to a type of artificial intelligence (AI) that is designed to perform any intellectual task that a human being can. It is a broad, general-purpose AI system that can reason, learn, and apply knowledge across a wide range of domains, tasks, and situations.\n\nGen AI is often described as a "Swiss Army knife" of AI systems, as it is designed to be versatile and adaptable to different tasks and environments. It is typically based on a combination of machine learning, deep learning, and symbolic AI approaches, and is often trained on large datasets and massive amounts of computational power.\n\nSome key characteristics of Gen AI include:\n\n1. **Generalizability**: Gen AI is designed to perform well across a wide range of tasks and domains, without requiring extensive retraining or fine-tuning.\n2. **Reasoning and problem-solving**: Gen AI is capable of reasoning, problem-solving, and decision-making, often using a combination of logical and pr

# Embedding

In [10]:
from langchain_cohere import CohereEmbeddings

embedding = CohereEmbeddings(
    model = "embed-english-v3.0",
    cohere_api_key= cohere_api_key
)

# Load the db

In [15]:
from langchain.vectorstores import FAISS
vectore = FAISS.load_local('faiss_db',embeddings=embedding,allow_dangerous_deserialization=True)

In [18]:
vectore.similarity_search('Frameworks - Libraries')

[Document(id='5c2f281e-6f31-473a-bfdd-ce0cff75efeb', metadata={}, page_content='–Frameworks - Libraries:Streamlit, FastAPI, Flask, Pandas, Numpy\n–Web: HTML, CSS'),
 Document(id='57a00402-f092-4446-9e22-74ec3bd02091', metadata={}, page_content='∗ Built an intelligent agent framework usingLangGraph and integrated Groq-hosted Gemma-9B LLM,'),
 Document(id='af7f56c3-a36f-4790-9797-77bba1fa9e06', metadata={}, page_content='∗ Tech stack: Python, Streamlit, FastAPI, LangChain, FAISS, Cohere LLMs, OpenAI API, Gemini'),
 Document(id='1c3a7c01-58e7-4907-beae-4baebbf262fc', metadata={}, page_content='∗ Tech stack: Python, FastAPI, Streamlit, LangGraph, LangChain, Groq LLM (Gemma-9B), Logging')]

In [19]:
# Load the pdf
from langchain.document_loaders import PyPDFLoader
loader = PyPDFLoader('EMPLOYEE_AGREEMENT.pdf')
loaded = loader.load()
len(loaded)

13

In [20]:
loaded[0]

Document(metadata={'producer': 'Skia/PDF m123', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'creationdate': '2024-04-03T12:51:00+00:00', 'title': 'EMPLOYEE AGREEMENT', 'moddate': '2024-04-03T12:51:00+00:00', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1'}, page_content='EX-10.1 3 smtp_ex10z1.htm EMPLOYEE AGREEMENT\nEXHIBIT 10.1\nEMPLOYEE AGREEMENT\nTHIS EMPLOYEE AGREEMENT made as of September___, 2014, by and between SharpSpring,\nInc., a Delaware corporation (the “Company”), whose principal place of business is at 802 NW 5th Avenue,\nSuite 100, Gainesville FL 32601; and Richard Carlson (“Employee”). This Employee Agreement replaces in\nits entirety the employee agreement dated August 15, 2014 between Employee and the Company. \nWHEREAS, the Company wishes to procure the services of Employee under the terms and\nconditions set forth and Employee wishes to be employed o

In [21]:
# chunking
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 150
)

In [22]:
chunks = text_splitter.split_documents(loaded)

In [23]:
len(chunks)

41

In [26]:
print(chunks[0].page_content)

EX-10.1 3 smtp_ex10z1.htm EMPLOYEE AGREEMENT
EXHIBIT 10.1
EMPLOYEE AGREEMENT
THIS EMPLOYEE AGREEMENT made as of September___, 2014, by and between SharpSpring,
Inc., a Delaware corporation (the “Company”), whose principal place of business is at 802 NW 5th Avenue,
Suite 100, Gainesville FL 32601; and Richard Carlson (“Employee”). This Employee Agreement replaces in
its entirety the employee agreement dated August 15, 2014 between Employee and the Company. 
WHEREAS, the Company wishes to procure the services of Employee under the terms and
conditions set forth and Employee wishes to be employed on these terms and conditions.
WHEREAS, the parties to this Employee Agreement wish to enter into a written expression of their
relationship as Employer and Employee.
THEREFORE, in consideration of the agreements contained in this Employee Agreement, the
parties, intending to be legally bound, agree as follows:
ARTICLE 1
Employment


# step -3 Embedding

# step create vectore database

In [27]:
from langchain.vectorstores import FAISS

In [28]:
vectore = FAISS.from_documents(chunks,embedding)

# Retriver

In [85]:
retriever = vectore.as_retriever(search_kwargs={"k": 5})


In [87]:
res = retriever.get_relevant_documents('Assignment of Inventions and Works Made for Hire')

In [88]:
res

[Document(id='32b65bab-d684-4d3b-866e-4eeddb7c3bc9', metadata={'producer': 'Skia/PDF m123', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'creationdate': '2024-04-03T12:51:00+00:00', 'title': 'EMPLOYEE AGREEMENT', 'moddate': '2024-04-03T12:51:00+00:00', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'total_pages': 13, 'page': 4, 'page_label': '5'}, page_content='8.3 Assignment of Inventions and Works Made for Hire.   Employee hereby irrevocably assigns and\ntransfers, and agrees to assign and transfer, to the Company all of Employee’s right, title and interest in and\nto any and all Inventions and Works Made for Hire (each as hereinafter defined) made, generated or\nconceived by Employee while employed by the Company at any time, whether alone or with the assistance of\nothers, whether or not made, generated or conceived during normal business hours, and whether or not his\nemployment with the Company is hereafter ter

In [34]:
print(res[0].page_content)

8.3 Assignment of Inventions and Works Made for Hire.   Employee hereby irrevocably assigns and
transfers, and agrees to assign and transfer, to the Company all of Employee’s right, title and interest in and
to any and all Inventions and Works Made for Hire (each as hereinafter defined) made, generated or
conceived by Employee while employed by the Company at any time, whether alone or with the assistance of
others, whether or not made, generated or conceived during normal business hours, and whether or not his
employment with the Company is hereafter terminated for any reason whatsoever. For purposes of this
Employee Agreement, “Inventions” shall mean any and all discoveries, improvements, innovations, ideas,
formulae, devices, systems, software programs, processes, products and any other creations similar thereto
which pertain or relate to the Company’s systems and technologies that enable: marketing automation, call


# RetrivalQna chain

In [ ]:
from langchain.chains import RetrievalQA
retrival_qa = RetrievalQA.from_chain_type(
    llm = llm,
    chain_type = 'stuff',
    retriever = retriver
)

In [38]:
res = retrival_qa.invoke("Tell me about Assignment of Inventions and Works Made for Hire")

In [40]:
print(res['result'])

The "Assignment of Inventions and Works Made for Hire" is a provision in an employee agreement that outlines the terms under which an employee assigns their intellectual property rights to their employer.

In this specific provision, the employee agrees to assign all of their rights, title, and interest in and to any inventions and works made for hire that they create while employed by the company, regardless of whether they are created during normal business hours or not. This includes:

1. Inventions: Any and all discoveries, improvements, innovations, ideas, formulae, devices, systems, software programs, processes, products, and other creations that pertain or relate to the company's systems and technologies.
2. Works Made for Hire: Any and all work made for hire, as defined in Section 101 of the United States Copyright Law, Title 17 of the United States Code, as amended.

The company has the right to request that the employee executes and signs any applications, assignments, and ot

In [41]:
retrival_qa.invoke("what is RAG")

{'query': 'what is RAG', 'result': "I don't know."}

In [42]:
# Multiple Query Retriever

In [43]:
from langchain.retrievers import MultiQueryRetriever
chain = MultiQueryRetriever.from_llm(
    llm = llm,
    retriever= retriver
)

In [47]:
chain.invoke('Assignment of Inventions and Works Made for Hire')

[Document(id='b4519d2a-6a9f-454d-83c2-6810ac4f9102', metadata={'producer': 'Skia/PDF m123', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'creationdate': '2024-04-03T12:51:00+00:00', 'title': 'EMPLOYEE AGREEMENT', 'moddate': '2024-04-03T12:51:00+00:00', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'total_pages': 13, 'page': 6, 'page_label': '7'}, page_content='performed any and all duties or obligations that he may have under any contract or agreement with a former\nEmployer or other party, including, without limitation, the return of all confidential materials; and (iii)\nEmployee is currently not in possession of any confidential materials or property belonging to any such\nformer Employer or other party.  Employee acknowledges and agrees that he shall advise the Company in the\nevent that his duties with the Company should be changed or enlarged in such a manner as to conflict with\nany such prior contract, agree

In [48]:
import logging
logging.basicConfig()
logging.getLogger('langchain.retrievers.multi_query').setLevel(logging.INFO)

In [49]:
res = chain.invoke('Assignment of Inventions and Works Made for Hire')
len(res)

INFO:langchain.retrievers.multi_query:Generated queries: ['Here are three different versions of the given user question to retrieve relevant documents from a vector database:', ' ', '1. What are the laws and regulations governing the assignment of inventions and works made for hire, including ownership and intellectual property rights?', ' ', '2. Identify documents related to intellectual property law, specifically those discussing the transfer of ownership and rights for inventions and works created under a work-for-hire agreement.', ' ', '3. Retrieve documents that discuss the contractual and legal frameworks surrounding the assignment of inventions and works made for hire, including the roles of employers, employees, and independent contractors.']


12

# Prompt with LLM Call with Vectore database

In [50]:
from langchain.prompts import PromptTemplate
template = """" 
You are a inteligent assistant retrieving accurate information.
- answer on;y using given context
- if context is not relevant to the question , say "Don't have context".

context :{context}
user questions :{question}

Your answer :


"""

In [51]:
prompt = PromptTemplate(
    input_variables=['context','question'],
    template= template
)

In [ ]:
prompt = PromptTemplate.from_template(
    """ 
You are a inteligent assistant retrieving accurate information.
- answer on;y using given context
- if context is not relevant to the question , say "Don't have context".

context :{context}
user questions :{question}

Your answer :


"""
)

In [55]:
# Runnable pass thorugh
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

In [56]:
questions_only = RunnableLambda(lambda x : x['question'])

In [57]:
chain = (
    {
        'context': questions_only | retriver,
        'question' : RunnablePassthrough()
    }
    | prompt
    |llm
)

In [58]:
res = chain.invoke({'question':'Tell me about Assignment of Inventions and Works Made for Hire'})

In [59]:
res

AIMessage(content="According to the provided context, specifically from Document(id='32b65bab-d684-4d3b-866e-4eeddb7c3bc9') and Document(id='94876c59-e37d-4146-b6e1-c3aa194fbb2c'), the Employee hereby irrevocably assigns and transfers all of their right, title, and interest in and to any and all Inventions and Works Made for Hire to the Company.\n\nInventions are defined as any and all discoveries, improvements, innovations, ideas, formulae, devices, systems, software programs, processes, products, and any other creations similar thereto which pertain or relate to the Company's systems and technologies that enable: marketing automation, call tracking, customer relationship management, sales automation, and/or email delivery.\n\nWorks Made for Hire are defined as any and all “work made for hire”, as that term is defined in Section 101 of the United States Copyright Law, Title 17 of the United States Code, as amended.\n\nThe Employee agrees to execute and sign any and all applications, a

In [60]:
print(res.content)

According to the provided context, specifically from Document(id='32b65bab-d684-4d3b-866e-4eeddb7c3bc9') and Document(id='94876c59-e37d-4146-b6e1-c3aa194fbb2c'), the Employee hereby irrevocably assigns and transfers all of their right, title, and interest in and to any and all Inventions and Works Made for Hire to the Company.

Inventions are defined as any and all discoveries, improvements, innovations, ideas, formulae, devices, systems, software programs, processes, products, and any other creations similar thereto which pertain or relate to the Company's systems and technologies that enable: marketing automation, call tracking, customer relationship management, sales automation, and/or email delivery.

Works Made for Hire are defined as any and all “work made for hire”, as that term is defined in Section 101 of the United States Copyright Law, Title 17 of the United States Code, as amended.

The Employee agrees to execute and sign any and all applications, assignments, and other doc

# Easy way to pass 

In [74]:
from langchain.prompts import PromptTemplate
template = """" 
You are a inteligent assistant retrieving accurate information.
- answer on;y using given context
-  **answer should be bullter format**
- if context is not relevant to the question , say "Don't have context".

context :{context}
user questions :{question}

Your answer :


"""

In [75]:
prompt = PromptTemplate(
    input_variables=['context','question'],
    template= template
)

# use memeroy

In [76]:
from langchain.memory import ConversationBufferMemory

In [77]:
# using chain
from langchain.chains import ConversationalRetrievalChain
chain = ConversationalRetrievalChain.from_llm(
    llm = llm,
    retriever = retriver,
    memory = ConversationBufferMemory(memory_key='chat_history',return_messages=True),
    combine_docs_chain_kwargs = {'prompt':prompt}
)

In [78]:
res = chain.invoke('Tell me about Assignment of Inventions and Works Made for Hire')

In [79]:
print(res['answer'])

**BULLET POINT ANSWER**

- **Assignment of Inventions**: 
  - Employee assigns all rights, title, and interest to the Company for any inventions made while employed.
  - Includes inventions made alone or with the assistance of others, during or outside normal business hours.
  - Inventions pertain to marketing automation, call tracking, customer relationship management, sales automation, and/or email delivery.

- **Works Made for Hire**: 
  - Defined as works made for hire under the United States Copyright Law, Title 17 of the United States Code.
  - Includes any work created by the Employee while employed by the Company.
  - Employee must execute and sign applications, assignments, and other documents as requested by the Company to obtain intellectual property protection.
